Dataset: https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database

Citation: -M.E.H. Chowdhury, T. Rahman, A. Khandakar, R. Mazhar, M.A. Kadir, Z.B. Mahbub, K.R. Islam, M.S. Khan, A. Iqbal, N. Al-Emadi, M.B.I. Reaz, M. T. Islam, “Can AI help in screening Viral and COVID-19 pneumonia?” IEEE Access, Vol. 8, 2020, pp. 132665 - 132676. Paper link
-Rahman, T., Khandakar, A., Qiblawey, Y., Tahir, A., Kiranyaz, S., Kashem, S.B.A., Islam, M.T., Maadeed, S.A., Zughaier, S.M., Khan, M.S. and Chowdhury, M.E., 2020. 

Goal: Create a model with reasonable performance that predicts whether patient has COVID-19 based on their Chest X-ray

Acknowledgements: Special thanks to the Udemy Course Deep Learning with PyTorch for Medical Image Analysis by Jose Portilla and his team. Also, Claude AI for providing insights and answering my questions on how to best achieve my goal and understand the processes along the way. 

Status: IN PROGRESS

In [16]:
#Importing packages

#need for data manipulation (merging datasets)
import pandas as pd
#merging folders 
import os
import shutil



In [ ]:
#Reading the csv file containing the results of Chest X-ray image

Positives=pd.read_excel(r"Folder\Folder\COVID.metadata.xlsx")

#Reading the normal chest x-ray data
Negatives=pd.read_excel(r"Folder\Folder\Normal.metadata.xlsx")


Section 2: To make it easier to run the model, we can merge both Negatives and Positives into one big dataset within the excel file. However, this is only the Excel file, we need to do the same for our chest-xray image files located in two folders. 

In [11]:
#See the first columns and rows to of the dataset to make sure our merge is clean

Positives.head()


,FILE NAME,FORMAT,SIZE,URL
0,COVID-1,PNG,256*256,https://sirm.org/category/senza-categoria/covi...
1,COVID-2,PNG,256*256,https://sirm.org/category/senza-categoria/covi...
2,COVID-3,PNG,256*256,https://sirm.org/category/senza-categoria/covi...
3,COVID-4,PNG,256*256,https://sirm.org/category/senza-categoria/covi...
4,COVID-5,PNG,256*256,https://sirm.org/category/senza-categoria/covi...


In [10]:
#See the first columns and rows to of the dataset to make sure our merge is clean
Negatives.head()

,FILE NAME,FORMAT,SIZE,URL
0,NORMAL-1,PNG,256*256,https://www.kaggle.com/c/rsna-pneumonia-detect...
1,NORMAL-2,PNG,256*256,https://www.kaggle.com/c/rsna-pneumonia-detect...
2,NORMAL-3,PNG,256*256,https://www.kaggle.com/c/rsna-pneumonia-detect...
3,NORMAL-4,PNG,256*256,https://www.kaggle.com/c/rsna-pneumonia-detect...
4,NORMAL-5,PNG,256*256,https://www.kaggle.com/c/rsna-pneumonia-detect...


In [12]:
#lists the number of rows so we can double check after merge that we have all the rows if we are doing a 
# a data stack on postives and negatives. So should be 3616 + 10192=13808

print("Positive rows:", len(Positives))
print("Negative rows:", len(Negatives))


Positives rows: 3616
Negatives rows: 10192


In [15]:
#Both have the same column names thus, we can merge without data cleaning
Merge= pd.concat([Positives,Negatives], ignore_index=True)
#prints out the shape or rows,columns. We can see we have 13,808 rows and 4 columns which shows a clean
#stacking of the rows from positives and negatives

print(Merge.shape)

(13808, 4)


In [ ]:
#Merging two folders

#location of our Positive Chest X-ray Folder
Positive_Folder= r"Folder\Folder\COVID\images"
#location of our Negative Chest X-ray Folder
Negative_Folder= r"Folder\Folder\Normal\images"

#Location of where we wish to put our output folder
Output_Folder= r"Folder\Folder\Merged"

#this code creates the Output_Folder if it doesn't exist. Exist_ok=True means don't do anything if it realdy exists because without this code,
#we would get an error
os.makedirs(Output_Folder,exist_ok=True)

#setting up our lists. So that if a file gets skipped or copied, it is inputted into these lists 

skipped = []

copied = []



Section 2: We will set up the file pathways of how to import and export the X-ray images. The goal is to connect the patient number from our Merge excel to it's chest x-ray image. In this case, we don't have patient numbers, so we will presume each row is a unique patient id. 

For example, if the 1st row is Normal-1 in the Merge Dataset then, we can label their chest x-ray image from another folder as Normal_Patient_ID_1. 

In [ ]:
# Setup
#The path where our chest x-ray images located
Origin_Path=




#Location of the drive for the images

Section 3: Normalizing the pixels of Chest X-ray images using Z formula. 

Each Chest X-ray can have different brightness due to the different machines used. By standardizing each chest x-ray, we can reduce Bias. For this section, we need to find the mean and standard deviation to  plug into a transform function (next section) and have each image be loaded and standardize. The Z formula which the transform function does in the backend is shown: 

           X-μ​
      Z=  -----
            σ

For example: if we have one chest x-ray image that consists of multiple pixel values 

      0.3  0.5  0.8  0.6
      0.4  0.7  0.5  0.3
      0.6  0.5  0.4  0.7
      0.8  0.3  0.6  0.5

Then, after standardization it would change to: 
 
        -1.0  0.0   1.5  0.5
        -0.5  1.0   0.0 -1.0
         0.5  0.0  -0.5  1.0
         1.5 -1.0   0.5  0.0

If we focus on how pixel number 0.3 -> -1.0 (upper left corner) with mean=0.5 and standard deviation 0.2 we would have formula below: 

Z= (0.3-0.5)/0.2

Z= -1.0

Standardization is helpful as our main goal is to see differences of pixel values for each Chest X-ray image (something that's hard to see with the naked eye for health professionals). So, we can say that Healthy lung pixels have low z-score with low or no normal brightness (Z~0). Whereas, those with bright pixels with high Z-score could be those with a positive chest X-ray. 